In [ ]:
import requests
from requests.packages.urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
import csv

In [ ]:
class TimeoutHttpAdapter(HTTPAdapter):
    def __init__(self, timeout=None, *args, **kwargs):
        self.timeout = timeout
        if "timeout" in kwargs:
            del kwargs["timeout"]
        super().__init__(*args, **kwargs)

    def send(self, *args, **kwargs):
        kwargs['timeout'] = self.timeout
        return super().send(*args, **kwargs)

In [ ]:
r=requests.Session()
retry_strategy = Retry(
            total=3,
            status_forcelist=[104, 429, 500, 502, 503, 504],
            allowed_methods=["HEAD", "GET" "POST", "PUT", "DELETE", "OPTIONS", "TRACE"],
            backoff_factor=2
        )
r.headers["User-Agent"]='My User Agent 1.0'
r.mount('https://', TimeoutHttpAdapter(timeout=None, max_retries=retry_strategy))
r.mount('http://', TimeoutHttpAdapter(timeout=None, max_retries=retry_strategy))

req = r.get('https://tilakmarg.com/forum')

In [ ]:
soup = BeautifulSoup(req.content, 'html.parser')

In [ ]:
s = soup.find_all('li', class_='bbp-forum-info')

In [ ]:
link_list=[]
for li in s:
    a = li.find("a")
    if a:
        link_list.append(a.attrs["href"])


In [ ]:
link_list

In [ ]:
gcnt=0
rows=[]
for link in link_list:
    temp=link
    while (True):
        req = r.get(temp)
        soup = BeautifulSoup(req.content.decode(), "html.parser")
        query_link=(soup.find_all('a',class_='bbp-topic-permalink'))
        for qlink in query_link:
            qreq=r.get(qlink.attrs['href'])
            qsoup=BeautifulSoup(qreq.content.decode(), "html.parser")
            thread=qsoup.find_all('div',class_='bbp-reply-content')
            qheading=qsoup.find('h1',class_='entry-title td-page-title').text
            cnt=0
            query_thread=[]
            for item in thread:
                if(item.find('p') is not None):
                    paragraph=''
                    for para in item.find_all('p'):
                        paragraph+=para.text
                    query_thread.append(paragraph)
                if cnt>=2:
                    break
                cnt+=1
            if len(query_thread)>=2:
                rows.append([qheading.strip(),query_thread[0].strip(),query_thread[1].strip()])
        nxtpage = soup.find("a", class_="next page-numbers")
        if nxtpage is None:
            break
        temp=nxtpage.attrs['href']
        print(nxtpage)
    gcnt+=1

In [ ]:
len(rows)

In [ ]:
rows[-1]

In [ ]:
fields = ['title', 'question', 'answer'] 
with open('legal_queries.csv', 'w+') as f:
      
    # using csv.writer method from CSV package
    write = csv.writer(f)
      
    write.writerow(fields)
    write.writerows(rows)

In [ ]:
f.close()